In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [2]:
class Layer:
    def forward(self, X):
        raise NotImplementedError

    def backward(self, dY):
        raise NotImplementedError

    def update(self, lr):
        pass

class Dense(Layer):
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * 0.01
        self.b = np.zeros((1, out_features))

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, dZ):
        m = self.X.shape[0]
        self.dW = self.X.T @ dZ
        self.db = np.sum(dZ, axis=0, keepdims=True)
        return dZ @ self.W.T

    def update(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db
      
class ReLU(Layer):
    def forward(self, Z):
        self.Z = Z
        return np.maximum(0, Z)

    def backward(self, dA):
        return dA * (self.Z > 0)
    
class MSELoss:
    def forward(self, Y_hat, Y):
        self.Y_hat = Y_hat
        self.Y = Y
        return np.mean((Y_hat - Y) ** 2)

    def backward(self):
        m = self.Y.shape[0]
        return 2 * (self.Y_hat - self.Y) / m    

class SoftmaxCrossEntropy:
    def forward(self, Z, Y):
        self.Y = Y

        exp = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        self.Y_hat = exp / np.sum(exp, axis=1, keepdims=True)

        loss = -np.mean(np.sum(Y * np.log(self.Y_hat + 1e-9), axis=1))
        return loss

    def backward(self):
        return self.Y_hat - self.Y


class sigmoid(Layer):
    def forward(self, z):
        self.a = 1 / (1 + np.exp(-z)) 
        return self.a
    
    def backward(self, da):
        return self.a * (1 - self.a) * da 
 

class Model:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, X):
        i = 0
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def update(self, lr):
        for layer in self.layers:
            layer.update(lr)

In [3]:
# Load dataset
X, y = load_iris(return_X_y=True)

# Normalize features
X = (X - X.mean(axis=0)) / X.std(axis=0)

# One-hot encode labels
Y = np.eye(3)[y]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, shuffle=True, random_state=1)


In [4]:
np.random.seed(0)

model = Model()
model.add(Dense(4, 12))
model.add(sigmoid())
model.add(Dense(12, 3))  # Linear output
model.add(sigmoid())

loss_fn = MSELoss()


In [5]:
lr = 0.09
epochs = 20000

for epoch in range(epochs):
    # Forward
    outputs = model.forward(X)
    loss = loss_fn.forward(outputs, Y)

    # Backward
    grad = loss_fn.backward()
    model.backward(grad)

    # Update
    model.update(lr)

    if epoch % 300 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.6f}")


Epoch 0, Loss: 0.248290
Epoch 300, Loss: 0.216463
Epoch 600, Loss: 0.129001
Epoch 900, Loss: 0.107570
Epoch 1200, Loss: 0.099670
Epoch 1500, Loss: 0.093619
Epoch 1800, Loss: 0.086857
Epoch 2100, Loss: 0.078426
Epoch 2400, Loss: 0.069779
Epoch 2700, Loss: 0.062332
Epoch 3000, Loss: 0.055768
Epoch 3300, Loss: 0.049551
Epoch 3600, Loss: 0.043546
Epoch 3900, Loss: 0.037936
Epoch 4200, Loss: 0.033000
Epoch 4500, Loss: 0.028902
Epoch 4800, Loss: 0.025631
Epoch 5100, Loss: 0.023071
Epoch 5400, Loss: 0.021073
Epoch 5700, Loss: 0.019501
Epoch 6000, Loss: 0.018248
Epoch 6300, Loss: 0.017235
Epoch 6600, Loss: 0.016401
Epoch 6900, Loss: 0.015706
Epoch 7200, Loss: 0.015117
Epoch 7500, Loss: 0.014613
Epoch 7800, Loss: 0.014176
Epoch 8100, Loss: 0.013794
Epoch 8400, Loss: 0.013456
Epoch 8700, Loss: 0.013156
Epoch 9000, Loss: 0.012887
Epoch 9300, Loss: 0.012644
Epoch 9600, Loss: 0.012423
Epoch 9900, Loss: 0.012222
Epoch 10200, Loss: 0.012038
Epoch 10500, Loss: 0.011869
Epoch 10800, Loss: 0.011713
Epoc

In [6]:
out = model.forward(X)
out = np.round(out)
accuracy = np.mean(out == Y)
print("Accuracy: ", accuracy)


Accuracy:  0.9866666666666667
